# nnU-Net BraTS 2024 MEN-RT — Google Colab Pipeline

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Run cells top to bottom in order
3. Keep browser tab open during training

**Edit ONLY Cell 3 (config). All other cells run as-is.**

| Cell | Step | Notes |
|---|---|---|
| 1 | Install packages | Once per session |
| 2 | GPU check | |
| 3 | **Config (edit here)** | Set NNUNET_NUM_EPOCHS here |
| 4 | Mount Drive | |
| 5 | Kaggle credentials | |
| 6 | Download data | |
| 7 | Clone repo + env | |
| 8 | Register trainer | |
| 9 | Prepare dataset | |
| 10 | Verify dataset | |
| 11 | Preprocessing | |
| 12 | **Helper functions** | Run once per session |
| 13 | **Fold 0 Training** | Saves checkpoints to Drive |
| 14 | Fold 0 Inference + Viz | Dice scores + overlays |
| 15 | **Fold 1 Training** | Saves checkpoints to Drive |
| 16 | Fold 1 Inference + Viz | |
| 17 | **Fold 2 Training** | Saves checkpoints to Drive |
| 18 | Fold 2 Inference + Viz | |
| 19 | **Fold 3 Training** | Saves checkpoints to Drive |
| 20 | Fold 3 Inference + Viz | |
| 21 | **Fold 4 Training** | Saves checkpoints to Drive |
| 22 | Fold 4 Inference + Viz | |
| 23 | Cross-fold Summary | All folds combined |

In [ ]:
# -- Cell 1: Install All Packages ------------------------------------------------
import subprocess, sys

packages = [
    'nnunetv2>=2.4',
    'nibabel>=5.0',
    'SimpleITK>=2.3',
    'scipy>=1.11',
    'scikit-image>=0.21',
    'medpy>=0.5',
    'surface-distance>=0.1',
    'pandas>=2.0',
    'matplotlib>=3.8',
    'seaborn>=0.13',
    'mlflow>=2.10',
    'python-dotenv>=1.0',
    'pyyaml>=6.0',
    'tqdm>=4.66',
    'loguru>=0.7',
    'rich>=13.0',
    'kaggle',
]

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    check=True
)
print('All packages installed successfully.')


In [ ]:
# -- Cell 2: Verify GPU ----------------------------------------------------------
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

if not torch.cuda.is_available():
    raise RuntimeError('ERROR: No GPU. Go to Runtime -> Change runtime type -> T4 GPU')

print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('GPU OK.')


In [ ]:
# -- Cell 3: Configuration -------------------------------------------------------
# EDIT ONLY THIS CELL.

KAGGLE_DATASET = 'maryampervaiz24029/meningioma-50-cases'
GITHUB_REPO    = 'https://github.com/maryampervaiz-alt/nnunet-training.git'

REPO_DIR     = '/content/nnunet'
DATA_DIR     = '/content/brats_data/BraTS-MEN-RT-Train-v2'
DRIVE_OUTPUT = '/content/drive/MyDrive/nnunet_checkpoints'

DATASET_ID       = '001'
DATASET_NAME     = 'BraTSMENRT'
LABEL_SUFFIX     = 'gtv'
MAX_TRAIN_CASES  = 20

NNUNET_CONFIGURATION = '3d_fullres'
NNUNET_TRAINER_CLASS = 'nnUNetTrainerEarlyStopping'
NNUNET_PLANS         = 'nnUNetPlans'
NUM_FOLDS            = 5
NNUNET_SEED          = 42
NNUNET_NUM_EPOCHS    = 3     # 3=pipeline test | 50=preliminary | 1000=full training
RESUME_TARGET_EPOCHS = 50

ES_PATIENCE  = 50
ES_MIN_DELTA = 0.0001
ES_WARMUP    = 50

CUDA_VISIBLE_DEVICES = '0'
NNUNET_N_PROC_DA     = '2'
NNUNET_COMPILE       = 'false'

INFERENCE_STEP_SIZE   = 0.5
INFERENCE_DISABLE_TTA = 'false'
INFERENCE_DEVICE      = 'cuda'

PP_THRESHOLDS      = '10,25,50,100,200,500'
NSD_TOLERANCE_MM   = 2.0
LOW_DICE_THRESHOLD = 0.5
BOOTSTRAP_N        = 2000

print('Configuration set.')
print(f'  Epochs per fold : {NNUNET_NUM_EPOCHS}')
print(f'  Max train cases : {MAX_TRAIN_CASES}')
print(f'  Folds           : {NUM_FOLDS}')
print(f'  Trainer         : {NNUNET_TRAINER_CLASS}')
print(f'  Drive output    : {DRIVE_OUTPUT}')


In [ ]:
# -- Cell 4: Mount Google Drive --------------------------------------------------
# IMPORTANT: if Drive was already mounted but files aren't saving,
# run: drive.mount('/content/drive', force_remount=True)

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

# --- Verify Drive is readable and writable ---
_drive_root = Path('/content/drive/MyDrive')
_out_dir    = Path(DRIVE_OUTPUT)
_out_dir.mkdir(parents=True, exist_ok=True)

# Write test
_test_file = _out_dir / '.write_test'
try:
    _test_file.write_text('ok')
    _test_file.unlink()
    _drive_ok = True
    print(f'Drive write test : PASSED')
except Exception as _e:
    _drive_ok = False
    print(f'Drive write test : FAILED — {_e}')
    print('ACTION: Try running this cell again with force_remount=True')

if _drive_ok:
    print(f'Drive output dir : {DRIVE_OUTPUT}')

    # Show what is already saved from previous sessions
    _subdirs = ['nnunet_results', 'metrics', 'visualizations', 'results']
    _any = False
    for _sub in _subdirs:
        _p = _out_dir / _sub
        if _p.exists():
            _files = list(_p.rglob('*'))
            _files = [f for f in _files if f.is_file()]
            if _files:
                _any = True
                _mb = sum(f.stat().st_size for f in _files) / 1e6
                print(f'  Drive/{_sub:<20} {len(_files):>4} files  ({_mb:.1f} MB)')
    if not _any:
        print('  (no previous session data found — fresh start)')
    print('Drive OK.')
else:
    raise RuntimeError(
        'Google Drive is not writable.\n'
        'Try: drive.mount(\'/content/drive\', force_remount=True)'
    )


In [ ]:
# -- Cell 5: Kaggle Credentials --------------------------------------------------
import os, json, shutil
from pathlib import Path

os.makedirs('/root/.kaggle', exist_ok=True)

try:
    from google.colab import userdata
    kaggle_content = userdata.get('kaggle_json')
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        f.write(kaggle_content)
    print('Kaggle credentials loaded from Colab secrets.')
except Exception:
    from google.colab import files
    print('Upload kaggle.json (kaggle.com -> Settings -> API -> Create Legacy API Key):')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    shutil.copy(fname, '/root/.kaggle/kaggle.json')
    print(f'Loaded from: {fname}')

os.chmod('/root/.kaggle/kaggle.json', 0o600)
with open('/root/.kaggle/kaggle.json') as f:
    info = json.load(f)
print(f'Kaggle username: {info["username"]}')
print('Kaggle credentials OK.')


In [ ]:
# -- Cell 6: Download BraTS Data -------------------------------------------------
import subprocess
from pathlib import Path

Path('/content/brats_data').mkdir(parents=True, exist_ok=True)

if Path(DATA_DIR).exists():
    cases = list(Path(DATA_DIR).iterdir())
    print(f'Data already exists: {len(cases)} items in {DATA_DIR}')
else:
    print(f'Downloading {KAGGLE_DATASET} ...')
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', KAGGLE_DATASET,
         '-p', '/content/brats_data', '--unzip'],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print('ERROR:', result.stderr)
        raise RuntimeError('Data download failed.')

if not Path(DATA_DIR).exists():
    print('Contents of /content/brats_data:')
    for p in Path('/content/brats_data').iterdir():
        print(f'  {p}')
    raise RuntimeError(f'Expected data at {DATA_DIR} not found. Update DATA_DIR in Cell 3.')

cases = sorted(Path(DATA_DIR).iterdir())
print(f'Data ready: {len(cases)} cases at {DATA_DIR}')


In [ ]:
# -- Cell 7: Clone Repository + Set Environment ----------------------------------
import subprocess, shutil, sys, os
from pathlib import Path

if Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
    print('Removed existing repo directory.')

result = subprocess.run(
    ['git', 'clone', GITHUB_REPO, REPO_DIR],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('ERROR:', result.stderr)
    raise RuntimeError('Git clone failed. Make sure the repo is public.')
print('Repository cloned.')

os.environ.update({
    'nnUNet_raw'             : '/content/nnunet_work/nnunet_raw',
    'nnUNet_preprocessed'    : '/content/nnunet_work/nnunet_preprocessed',
    'nnUNet_results'         : '/content/nnunet_work/nnunet_results',
    'DATASET_ID'             : DATASET_ID,
    'DATASET_NAME'           : DATASET_NAME,
    'LABEL_SUFFIX'           : LABEL_SUFFIX,
    'MAX_TRAIN_CASES'        : str(MAX_TRAIN_CASES),
    'TRAIN_SOURCE'           : DATA_DIR,
    'VAL_SOURCE'             : '',
    'NNUNET_CONFIGURATION'   : NNUNET_CONFIGURATION,
    'NNUNET_TRAINER_CLASS'   : NNUNET_TRAINER_CLASS,
    'NNUNET_PLANS_IDENTIFIER': NNUNET_PLANS,
    'NUM_FOLDS'              : str(NUM_FOLDS),
    'NNUNET_SEED'            : str(NNUNET_SEED),
    'NNUNET_NUM_EPOCHS'      : str(NNUNET_NUM_EPOCHS),
    'RESUME_TARGET_EPOCHS'   : str(RESUME_TARGET_EPOCHS),
    'ES_PATIENCE'            : str(ES_PATIENCE),
    'ES_MIN_DELTA'           : str(ES_MIN_DELTA),
    'ES_WARMUP'              : str(ES_WARMUP),
    'INFERENCE_STEP_SIZE'    : str(INFERENCE_STEP_SIZE),
    'INFERENCE_DISABLE_TTA'  : INFERENCE_DISABLE_TTA,
    'INFERENCE_DEVICE'       : INFERENCE_DEVICE,
    'PP_THRESHOLDS'          : PP_THRESHOLDS,
    'NSD_TOLERANCE_MM'       : str(NSD_TOLERANCE_MM),
    'LOW_DICE_THRESHOLD'     : str(LOW_DICE_THRESHOLD),
    'BOOTSTRAP_N'            : str(BOOTSTRAP_N),
    'CUDA_VISIBLE_DEVICES'   : CUDA_VISIBLE_DEVICES,
    'nnUNet_n_proc_DA'       : NNUNET_N_PROC_DA,
    'nnUNet_compile'         : NNUNET_COMPILE,
    'EXPERIMENT_NAME'        : 'BraTSMENRT_baseline',
    'MLFLOW_TRACKING_URI'    : '/content/nnunet_work/experiments/mlruns',
    'KAGGLE_OUTPUT_DIR'      : DRIVE_OUTPUT,
    'PYTHONUNBUFFERED'       : '1',
})

for d in [
    '/content/nnunet_work/nnunet_raw',
    '/content/nnunet_work/nnunet_preprocessed',
    '/content/nnunet_work/nnunet_results',
    '/content/nnunet_work/metrics',
    '/content/nnunet_work/results',
    '/content/nnunet_work/visualizations',
    '/content/nnunet_work/inference_outputs',
    '/content/nnunet_work/logs',
    '/content/nnunet_work/checkpoints',
    '/content/nnunet_work/predictions',
]:
    Path(d).mkdir(parents=True, exist_ok=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Working directory: {os.getcwd()}')
print('Environment configured.')


In [ ]:
# -- Cell 8: Register Custom nnU-Net Trainer -------------------------------------
import shutil, importlib.util, os
from pathlib import Path
import nnunetv2

trainer_src = Path(REPO_DIR) / 'src' / 'training' / 'nnunet_trainer_es.py'
if not trainer_src.exists():
    raise FileNotFoundError(f'Custom trainer not found: {trainer_src}')

nnunet_pkg  = Path(nnunetv2.__file__).parent
trainer_dst = nnunet_pkg / 'training' / 'nnUNetTrainer' / 'nnUNetTrainerEarlyStopping.py'
shutil.copy2(trainer_src, trainer_dst)

spec = importlib.util.spec_from_file_location('nnUNetTrainerEarlyStopping', trainer_dst)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
if not hasattr(mod, 'nnUNetTrainerEarlyStopping'):
    raise ImportError('nnUNetTrainerEarlyStopping class not found in trainer file')

print(f'Custom trainer registered at: {trainer_dst}')
print('Trainer registration OK.')


In [ ]:
# -- Cell 9: Prepare Dataset -----------------------------------------------------
import subprocess, sys, os
from pathlib import Path

print(f'Source : {os.environ["TRAIN_SOURCE"]}')
print(f'Output : {os.environ["nnUNet_raw"]}')
print(f'Cases  : {os.environ["MAX_TRAIN_CASES"]}')
print()

proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/01_prepare_dataset.py',
     '--train',     os.environ['TRAIN_SOURCE'],
     '--skip-val',
     '--log-dir',   '/content/nnunet_work/logs',
     '--max-cases', os.environ['MAX_TRAIN_CASES']],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Dataset preparation failed (rc={proc.returncode})')
print('\nDataset preparation complete.')


In [ ]:
# -- Cell 10: Verify Dataset Structure -------------------------------------------
import json, os
from pathlib import Path

ds_dir = Path(os.environ['nnUNet_raw']) / 'Dataset{:03d}_{}'.format(
    int(os.environ['DATASET_ID']), os.environ['DATASET_NAME'])
images = sorted((ds_dir / 'imagesTr').glob('*.nii.gz'))
labels = sorted((ds_dir / 'labelsTr').glob('*.nii.gz'))

print(f'Dataset dir : {ds_dir}')
print(f'Images      : {len(images)}')
print(f'Labels      : {len(labels)}')

if len(images) == 0:
    raise RuntimeError('No images found. Check Cell 9 output for errors.')
if len(images) != len(labels):
    raise RuntimeError(f'Image/label mismatch: {len(images)} vs {len(labels)}')

ds_json = ds_dir / 'dataset.json'
if ds_json.exists():
    d = json.loads(ds_json.read_text())
    print(f'dataset.json: {d.get("numTraining")} training cases')

print('Dataset verification OK.')


In [ ]:
# -- Cell 11: nnU-Net Planning & Preprocessing -----------------------------------
# --np 1 avoids Colab RAM crash. Only affects preprocessing parallelism.
import subprocess, sys, os

print('Running nnUNetv2_plan_and_preprocess ...')
proc = subprocess.Popen(
    [sys.executable, '-u', 'scripts/02_preprocess.py',
     '--configurations', '3d_fullres',
     '--np', '1',
     '--log-dir', '/content/nnunet_work/logs'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Preprocessing failed (rc={proc.returncode})')
print('\nPreprocessing complete.')


In [ ]:
# -- Cell 11b: Prepare Truly Held-Out External Test Set --------------------------
# Builds imagesTs (+ optional labelsTs_gt) from source cases that were NOT used
# in the 5-fold CV training pool.

import os, json, shutil
from pathlib import Path

SEP = '=' * 68
print(SEP)
print('  HELD-OUT EXTERNAL TEST SET PREPARATION')
print(SEP)

_ds = f"Dataset{int(os.environ['DATASET_ID']):03d}_{os.environ['DATASET_NAME']}"
ds_dir = Path(os.environ['nnUNet_raw']) / _ds
src_path = Path(os.environ['TRAIN_SOURCE'])

print(f'  Source path  : {src_path}')
print(f'  Dataset path : {ds_dir}')

if not src_path.exists():
    raise RuntimeError(f'TRAIN_SOURCE not found: {src_path}')
if not src_path.is_dir():
    raise RuntimeError(
        'TRAIN_SOURCE is not a directory. For held-out extraction, point TRAIN_SOURCE '
        'to an extracted BraTS case-folder directory.'
    )

imagesTr = ds_dir / 'imagesTr'
if not imagesTr.exists():
    raise RuntimeError('imagesTr missing. Run Cell 9 and Cell 10 first.')

train_case_ids = {p.name.replace('_0000.nii.gz', '') for p in imagesTr.glob('*_0000.nii.gz')}
all_case_ids = sorted([d.name for d in src_path.iterdir() if d.is_dir()])
held_out_ids = sorted(set(all_case_ids) - train_case_ids)

print(f'  Cases in source        : {len(all_case_ids)}')
print(f'  Cases in CV pool       : {len(train_case_ids)}')
print(f'  Held-out external test : {len(held_out_ids)}')

HELD_OUT_READY = False

if not held_out_ids:
    print('\nNo held-out cases found. Reduce MAX_TRAIN_CASES in Cell 6 to keep external cases.')
else:
    label_suffix = os.environ.get('LABEL_SUFFIX', 'gtv')
    imagesTs = ds_dir / 'imagesTs'
    labelsTs = ds_dir / 'labelsTs_gt'
    imagesTs.mkdir(parents=True, exist_ok=True)
    labelsTs.mkdir(parents=True, exist_ok=True)

    # Clean previous external set files to avoid stale leftovers.
    for old in imagesTs.glob('*.nii.gz'):
        old.unlink()
    for old in labelsTs.glob('*.nii.gz'):
        old.unlink()

    img_suffix = 't1c'
    for cand in ['t1c', 't1ce', 't1n', 't1', 'flair']:
        if list(src_path.glob(f'*/*_{cand}.nii.gz')) or list(src_path.glob(f'*/*_{cand}.nii')):
            img_suffix = cand
            break

    n_img, n_lbl = 0, 0
    print(f"\n  {'Case ID':<40}  {'Image':>7}  {'Label':>7}")
    print(f"  {'-'*59}")

    for case_id in held_out_ids:
        case_dir = src_path / case_id
        img_files = list(case_dir.glob(f'*_{img_suffix}.nii.gz')) + list(case_dir.glob(f'*_{img_suffix}.nii'))
        lbl_files = list(case_dir.glob(f'*_{label_suffix}.nii.gz')) + list(case_dir.glob(f'*_{label_suffix}.nii'))

        img_ok = bool(img_files)
        lbl_ok = bool(lbl_files)

        if img_ok:
            shutil.copy2(img_files[0], imagesTs / f'{case_id}_0000.nii.gz')
            n_img += 1
        if lbl_ok:
            shutil.copy2(lbl_files[0], labelsTs / f'{case_id}.nii.gz')
            n_lbl += 1

        print(f"  {case_id:<40}  {'OK' if img_ok else 'MISS':>7}  {'OK' if lbl_ok else 'MISS':>7}")

    (ds_dir / 'held_out_test_cases.json').write_text(json.dumps(held_out_ids, indent=2))

    HELD_OUT_READY = n_img > 0
    print(f"  {'-'*59}")
    print(f'  Copied images : {n_img}')
    print(f'  Copied labels : {n_lbl}')
    if n_img == 0:
        print('  ERROR: no held-out images copied.')
    elif n_lbl == 0:
        print('  WARNING: labels missing, so evaluation step will be skipped.')
    else:
        print('  Held-out external set ready. Run Cell 22b after all folds finish.')
print(SEP)

In [ ]:
# -- Cell 12: Helper Functions ---------------------------------------------------
# Run this ONCE before any fold cell.
import os, shutil, json, subprocess, sys
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from IPython.display import display, Image as IPImage

TRAINER     = os.environ['NNUNET_TRAINER_CLASS']
PLANS       = os.environ['NNUNET_PLANS_IDENTIFIER']
CONFIG      = os.environ['NNUNET_CONFIGURATION']
_did        = os.environ['DATASET_ID'].zfill(3)
_dname      = os.environ['DATASET_NAME']
DS          = f'Dataset{_did}_{_dname}'
RESULTS_DIR = Path(os.environ['nnUNet_results'])
METRICS_DIR = Path('/content/nnunet_work/metrics')
DRIVE_DIR   = Path(DRIVE_OUTPUT)

# ── Drive helpers ──────────────────────────────────────────────────────────────

def _check_drive_accessible():
    """Return True if Drive directory is writable."""
    test = DRIVE_DIR / '.access_test'
    try:
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        test.write_text('ok')
        test.unlink()
        return True
    except Exception as e:
        print(f'  WARNING: Drive not accessible: {e}')
        print('  Re-run Cell 4 to remount Drive.')
        return False


def save_fold_to_drive(fold):
    """Copy fold checkpoints + CSV to Drive with file-by-file verification."""
    if not _check_drive_accessible():
        print(f'  SKIPPED: Drive not writable. Run Cell 4 first.')
        return 0, 1

    src_dir = ckpt_dir(fold)
    dst_dir = (DRIVE_DIR / 'nnunet_results' / DS
               / f'{TRAINER}__{PLANS}__{CONFIG}' / f'fold_{fold}')
    dst_dir.mkdir(parents=True, exist_ok=True)

    SEP = '-' * 65
    print(f'\n  {"":=<65}')
    print(f'  DRIVE SAVE — Fold {fold}')
    print(f'  Source  : {src_dir}')
    print(f'  Dest    : {dst_dir}')
    print(f'  {SEP}')
    print(f'  {"File":<42} {"Size":>8}  Status')
    print(f'  {SEP}')

    n_ok, n_fail = 0, 0

    # Checkpoint files
    for pattern in ['*.pth', 'training_log*.txt', 'progress.png', 'debug.json']:
        for fp in sorted(src_dir.glob(pattern)):
            size_mb = fp.stat().st_size / 1e6
            dst_fp  = dst_dir / fp.name
            try:
                shutil.copy2(fp, dst_fp)
                if dst_fp.exists() and dst_fp.stat().st_size == fp.stat().st_size:
                    print(f'  {fp.name:<42} {size_mb:>6.2f}MB  OK')
                    n_ok += 1
                else:
                    print(f'  {fp.name:<42} {size_mb:>6.2f}MB  VERIFY FAILED (size mismatch)')
                    n_fail += 1
            except Exception as e:
                print(f'  {fp.name:<42} {size_mb:>6.2f}MB  ERROR: {e}')
                n_fail += 1

    # Training metrics CSV
    csv_src = METRICS_DIR / f'fold_{fold}_training.csv'
    if csv_src.exists():
        dm  = DRIVE_DIR / 'metrics'
        dm.mkdir(parents=True, exist_ok=True)
        csv_dst = dm / csv_src.name
        size_kb = csv_src.stat().st_size / 1e3
        try:
            shutil.copy2(csv_src, csv_dst)
            if csv_dst.exists():
                print(f'  {csv_src.name:<42} {size_kb:>6.1f}KB  OK')
                n_ok += 1
            else:
                print(f'  {csv_src.name:<42} {size_kb:>6.1f}KB  VERIFY FAILED')
                n_fail += 1
        except Exception as e:
            print(f'  {csv_src.name:<42}            ERROR: {e}')
            n_fail += 1
    else:
        print(f'  fold_{fold}_training.csv                             NOT FOUND (no metrics yet)')

    print(f'  {SEP}')
    if n_fail == 0:
        print(f'  RESULT: {n_ok} files saved to Drive successfully.')
    else:
        print(f'  RESULT: {n_ok} OK  |  {n_fail} FAILED  <- check Drive mount!')
    print(f'  {"":=<65}')
    return n_ok, n_fail


def drive_status():
    """Print a full table of everything currently saved on Drive."""
    SEP = '=' * 65
    print(f'\n{SEP}')
    print('  DRIVE STATUS CHECK')
    print(SEP)

    if not DRIVE_DIR.exists():
        print('  Drive directory not found. Run Cell 4 to mount Drive.')
        print(SEP)
        return

    # --- Checkpoints per fold ---
    print('\n  Fold Checkpoints (nnunet_results):')
    print(f'  {"Fold":<6} {"best.pth":>9} {"final.pth":>10} {"latest.pth":>11} {"log":>5} {"MB":>7}')
    print(f'  {"-"*52}')
    ckpt_root = DRIVE_DIR / 'nnunet_results'
    for fold in range(int(os.environ.get('NUM_FOLDS', 5))):
        fold_dir = (ckpt_root / DS / f'{TRAINER}__{PLANS}__{CONFIG}'
                    / f'fold_{fold}')
        if not fold_dir.exists():
            print(f'  {fold:<6} (not saved yet)')
            continue
        has_best   = (fold_dir / 'checkpoint_best.pth').exists()
        has_final  = (fold_dir / 'checkpoint_final.pth').exists()
        has_latest = (fold_dir / 'checkpoint_latest.pth').exists()
        n_logs     = len(list(fold_dir.glob('training_log*.txt')))
        total_mb   = sum(f.stat().st_size for f in fold_dir.rglob('*') if f.is_file()) / 1e6
        b = 'YES' if has_best   else 'NO '
        f = 'YES' if has_final  else 'NO '
        l = 'YES' if has_latest else 'NO '
        print(f'  {fold:<6} {b:>9} {f:>10} {l:>11} {n_logs:>5} {total_mb:>6.1f}')

    # --- Metrics CSVs ---
    print('\n  Metrics CSVs (Drive/metrics):')
    m_dir = DRIVE_DIR / 'metrics'
    if m_dir.exists():
        csvs = sorted(m_dir.glob('*.csv'))
        if csvs:
            for f in csvs:
                try:
                    rows = sum(1 for _ in open(f)) - 1
                except Exception:
                    rows = '?'
                size_kb = f.stat().st_size / 1e3
                print(f'  {f.name:<45} {rows:>5} rows  {size_kb:>6.1f} KB')
        else:
            print('  (none)')
    else:
        print('  (none)')

    # --- Visualizations ---
    print('\n  Visualizations (Drive/visualizations):')
    v_dir = DRIVE_DIR / 'visualizations'
    if v_dir.exists():
        pngs = list(v_dir.rglob('*.png'))
        print(f'  {len(pngs)} PNG file(s)')
        for fold in range(int(os.environ.get('NUM_FOLDS', 5))):
            fv = v_dir / f'fold_{fold}'
            if fv.exists():
                n = len(list(fv.glob('*.png')))
                print(f'    fold_{fold}: {n} overlays')
    else:
        print('  (none)')

    # --- Results ---
    print('\n  Evaluation Results (Drive/results):')
    r_dir = DRIVE_DIR / 'results'
    if r_dir.exists():
        rfiles = sorted(r_dir.glob('*'))
        if rfiles:
            for f in rfiles:
                if f.is_file():
                    print(f'  {f.name}  ({f.stat().st_size/1e3:.1f} KB)')
        else:
            print('  (none yet)')
    else:
        print('  (none yet)')

    print(f'\n{SEP}')


# ── Checkpoint helpers ─────────────────────────────────────────────────────────

def ckpt_dir(fold):
    return RESULTS_DIR / DS / f'{TRAINER}__{PLANS}__{CONFIG}' / f'fold_{fold}'


def is_fold_done(fold):
    return (ckpt_dir(fold) / 'checkpoint_final.pth').exists()


def restore_fold_from_drive(fold):
    src = (DRIVE_DIR / 'nnunet_results' / DS
           / f'{TRAINER}__{PLANS}__{CONFIG}' / f'fold_{fold}')
    if not src.exists():
        return False
    dst = ckpt_dir(fold)
    dst.mkdir(parents=True, exist_ok=True)
    restored = []
    for fp in list(src.glob('*.pth')) + list(src.glob('training_log*.txt')):
        shutil.copy2(fp, dst / fp.name)
        restored.append(fp.name)
    if restored:
        print(f'  Restored from Drive: {restored}')
    return bool(restored)


# ── Training log parsing ───────────────────────────────────────────────────────

def parse_training_log(fold):
    logs = sorted(ckpt_dir(fold).glob('training_log*.txt'))
    if not logs:
        return []
    rows = []
    epoch = loss = val_loss = dice = None
    for line in logs[-1].read_text().splitlines():
        line = line.strip()
        if ': ' not in line:
            continue
        msg = line.split(': ', 1)[1].strip()
        if msg.startswith('Epoch ') and len(msg) > 6:
            token = msg[6:].strip()
            if token.isdigit():
                if epoch is not None and loss is not None:
                    rows.append({'epoch': epoch, 'train_loss': loss,
                                 'val_loss': val_loss, 'pseudo_dice': dice})
                epoch = int(token)
                loss = val_loss = dice = None
        elif msg.startswith('train_loss '):
            try:
                loss = float(msg.split()[1])
            except Exception:
                pass
        elif msg.startswith('val_loss '):
            try:
                val_loss = float(msg.split()[1])
            except Exception:
                pass
        elif 'Pseudo dice' in msg and 'float32' in msg:
            start = msg.rfind('(')
            end   = msg.rfind(')')
            if start != -1 and end > start:
                try:
                    dice = float(msg[start + 1:end])
                except Exception:
                    pass
    if epoch is not None and loss is not None:
        rows.append({'epoch': epoch, 'train_loss': loss,
                     'val_loss': val_loss, 'pseudo_dice': dice})
    return rows


def print_fold_metrics(fold):
    rows = parse_training_log(fold)
    if not rows:
        print(f'  No training log found for fold {fold}')
        return
    valid_dices = [r['pseudo_dice'] for r in rows if r.get('pseudo_dice') is not None]
    best_dice   = max(valid_dices) if valid_dices else None
    print(f'\n  Fold {fold} — Epoch Log ({len(rows)} epochs):')
    print('  {:>6}  {:>12}  {:>10}  {:>12}'.format(
        'Epoch', 'Train Loss', 'Val Loss', 'Pseudo Dice'))
    print('  ' + '-' * 48)
    for r in rows:
        ep   = r.get('epoch', '?')
        l    = r.get('train_loss')  or float('nan')
        vl   = r.get('val_loss')    or float('nan')
        d    = r.get('pseudo_dice') or float('nan')
        mark = ' <- BEST' if (best_dice is not None and
                              not (d != d) and
                              abs(d - best_dice) < 1e-6) else ''
        print('  {:>6}  {:>12.4f}  {:>10.4f}  {:>12.4f}{}'.format(ep, l, vl, d, mark))
    if valid_dices:
        print(f'\n  Best Pseudo Dice  : {max(valid_dices):.4f}')
        print(f'  Final Pseudo Dice : {valid_dices[-1]:.4f}')
    print('  NOTE: Pseudo Dice is a training proxy. True Dice = inference cell.')


# ── CV split helpers ───────────────────────────────────────────────────────────

def get_val_cases(fold):
    splits = Path(os.environ['nnUNet_preprocessed']) / DS / 'splits_final.json'
    if not splits.exists():
        return []
    with open(splits) as f:
        data = json.load(f)
    return data[fold]['val'] if fold < len(data) else []


# ── Inference helpers ──────────────────────────────────────────────────────────

def run_inference_fold(fold):
    """Predict on the fold's validation split (held-out cases for that fold)."""
    val_cases = get_val_cases(fold)
    if not val_cases:
        print(f'  No validation cases for fold {fold}')
        return None
    raw_dir = Path(os.environ['nnUNet_raw']) / DS
    tmp_in  = Path(f'/content/nnunet_work/tmp_infer_fold{fold}')
    tmp_in.mkdir(parents=True, exist_ok=True)
    for case in val_cases:
        for fp in (raw_dir / 'imagesTr').glob(f'{case}*'):
            shutil.copy2(fp, tmp_in / fp.name)
    out_dir = Path('/content/nnunet_work/predictions') / f'fold_{fold}'
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'  Predicting {len(val_cases)} held-out validation cases ...')
    r = subprocess.run(
        ['nnUNetv2_predict', '-d', DS,
         '-i', str(tmp_in), '-o', str(out_dir),
         '-f', str(fold), '-tr', TRAINER, '-c', CONFIG],
        capture_output=True, text=True, env=os.environ.copy()
    )
    shutil.rmtree(tmp_in, ignore_errors=True)
    if r.returncode != 0:
        print(f'  Inference failed (rc={r.returncode}):')
        print(r.stderr[-2000:])
        return None
    preds = list(out_dir.glob('*.nii.gz'))
    print(f'  {len(preds)} predictions -> {out_dir}')
    return out_dir


def compute_dice(pred_path, gt_path):
    p     = np.asarray(nib.load(str(pred_path)).dataobj, dtype=np.uint8)
    g     = np.asarray(nib.load(str(gt_path)).dataobj,   dtype=np.uint8)
    tp    = int(np.logical_and(p > 0, g > 0).sum())
    denom = int((p > 0).sum()) + int((g > 0).sum())
    return 2 * tp / denom if denom > 0 else 1.0


def show_fold_predictions(fold, pred_dir):
    raw_dir   = Path(os.environ['nnUNet_raw']) / DS
    gt_dir    = raw_dir / 'labelsTr'
    img_dir   = raw_dir / 'imagesTr'
    val_cases = get_val_cases(fold)
    dices     = []
    for case in val_cases:
        pred_f = Path(pred_dir) / f'{case}.nii.gz'
        gt_f   = gt_dir / f'{case}.nii.gz'
        img_fs = sorted(img_dir.glob(f'{case}_0000*'))
        if not pred_f.exists() or not gt_f.exists() or not img_fs:
            print(f'  Skipping {case}: files missing')
            continue
        img  = nib.load(str(img_fs[0])).get_fdata()
        gt   = nib.load(str(gt_f)).get_fdata()
        pred = nib.load(str(pred_f)).get_fdata()
        gt_z = gt.sum(axis=(0, 1))
        z    = int(gt_z.argmax()) if gt_z.max() > 0 else img.shape[2] // 2
        d    = compute_dice(pred_f, gt_f)
        dices.append({'case': case, 'dice': d})
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        fig.suptitle(f'Fold {fold}  |  {case}  |  Dice = {d:.4f}', fontsize=12)
        ax[0].imshow(img[:, :, z].T, cmap='gray', origin='lower')
        ax[0].set_title('T1c Input'); ax[0].axis('off')
        ax[1].imshow(img[:, :, z].T, cmap='gray', origin='lower')
        mask_gt = np.ma.masked_where(gt[:, :, z].T == 0, gt[:, :, z].T)
        ax[1].imshow(mask_gt, cmap='Greens', alpha=0.5, origin='lower')
        ax[1].set_title('Ground Truth (GTV)'); ax[1].axis('off')
        ax[2].imshow(img[:, :, z].T, cmap='gray', origin='lower')
        mask_p = np.ma.masked_where(pred[:, :, z].T == 0, pred[:, :, z].T)
        ax[2].imshow(mask_p, cmap='Reds', alpha=0.5, origin='lower')
        ax[2].set_title(f'Prediction (Dice = {d:.3f})'); ax[2].axis('off')
        plt.tight_layout()
        vdir = DRIVE_DIR / 'visualizations' / f'fold_{fold}'
        vdir.mkdir(parents=True, exist_ok=True)
        save_path = vdir / f'{case}_z{z}.png'
        plt.savefig(str(save_path), dpi=110, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f'  {case}: Dice = {d:.4f}  [saved -> Drive]')
    return dices


def save_dice_csv(fold, dices):
    if not dices:
        return
    df  = pd.DataFrame(dices)
    dm  = DRIVE_DIR / 'metrics'
    dm.mkdir(parents=True, exist_ok=True)
    csv = dm / f'fold_{fold}_inference_dice.csv'
    df.to_csv(str(csv), index=False, float_format='%.6f')
    print(f'\n  Inference Dice — Fold {fold}  ({len(dices)} held-out cases):')
    print(f'  {"Metric":<12} {"Value":>8}')
    print(f'  {"-"*22}')
    print(f'  {"Mean":<12} {df["dice"].mean():>8.4f}')
    print(f'  {"Std":<12} {df["dice"].std():>8.4f}')
    print(f'  {"Median":<12} {df["dice"].median():>8.4f}')
    print(f'  {"Min":<12} {df["dice"].min():>8.4f}')
    print(f'  {"Max":<12} {df["dice"].max():>8.4f}')
    print(f'  CSV saved to Drive: {csv}')


# ── Per-fold orchestrators ─────────────────────────────────────────────────────

def _run_fold_training(fold):
    num_epochs = int(os.environ['NNUNET_NUM_EPOCHS'])
    SEP = '=' * 65
    print(SEP)
    print(f'  FOLD {fold}  |  TRAINING  |  {num_epochs} epochs')
    print(SEP)
    restore_fold_from_drive(fold)
    if is_fold_done(fold):
        print(f'Fold {fold} already complete (checkpoint_final.pth exists).')
        print_fold_metrics(fold)
        return
    has_latest = (ckpt_dir(fold) / 'checkpoint_latest.pth').exists()
    print('Resuming from checkpoint ...' if has_latest else 'Starting fresh ...')
    cmd = [
        sys.executable, '-u', 'scripts/03_train.py',
        '--folds',           str(fold),
        '--num-epochs',      str(num_epochs),
        '--seed',            os.environ['NNUNET_SEED'],
        '--trainer',         TRAINER,
        '--plans',           PLANS,
        '--es-patience',     os.environ['ES_PATIENCE'],
        '--es-min-delta',    os.environ['ES_MIN_DELTA'],
        '--es-warmup',       os.environ['ES_WARMUP'],
        '--log-dir',         '/content/nnunet_work/logs',
        '--metrics-dir',     '/content/nnunet_work/metrics',
        '--checkpoints-dir', '/content/nnunet_work/checkpoints',
    ]
    if has_latest:
        cmd.append('--continue-training')
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy()
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    print(SEP)
    if proc.returncode != 0:
        print(f'  ERROR: Fold {fold} training failed (rc={proc.returncode})')
        print(f'  Checkpoints NOT saved to Drive.')
        return
    print(f'  FOLD {fold} TRAINING COMPLETE')
    print(SEP)
    print_fold_metrics(fold)
    # Save to Drive with verification
    save_fold_to_drive(fold)


def _run_fold_inference(fold):
    SEP = '=' * 65
    print(SEP)
    print(f'  FOLD {fold}  |  INFERENCE (held-out validation cases)')
    print(SEP)
    if not (ckpt_dir(fold) / 'checkpoint_best.pth').exists():
        restore_fold_from_drive(fold)
    has_ckpt = ((ckpt_dir(fold) / 'checkpoint_best.pth').exists() or
                (ckpt_dir(fold) / 'checkpoint_final.pth').exists())
    if not has_ckpt:
        print(f'No checkpoint for fold {fold}. Run the training cell first.')
        return
    val_cases = get_val_cases(fold)
    print(f'Held-out validation cases ({len(val_cases)}): {val_cases}')
    pred_dir = run_inference_fold(fold)
    if pred_dir is None:
        return
    print(f'\nDice Scores (held-out cases — not seen during training):')
    dices = show_fold_predictions(fold, pred_dir)
    save_dice_csv(fold, dices)


print('Helper functions loaded.')
print('Run Cells 13-22 for per-fold training and inference.')
print('Run drive_status() at any time to check what is saved on Drive.')


In [ ]:
# -- Cell 12b: Drive Status Check ------------------------------------------------
# Run at any time to see EXACTLY what is saved on Drive.
# Especially useful after each fold training to confirm save was successful.
try:
    drive_status
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

drive_status()


In [ ]:
# -- Cell 13: Fold 0 Training ----------------------------------------------------
# Trains on ~16 cases, validates on ~4 held-out cases.
# After training: checkpoints + CSV saved to Drive with file-by-file verification.
# If training fails or Drive save fails — it will be clearly printed here.
try:
    _run_fold_training
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

_run_fold_training(fold=0)

print('\nDrive status after Fold 0 training:')
drive_status()


In [ ]:
# -- Cell 14: Fold 0 Inference + Visualization -----------------------------------
# Uses validation split from splits_final.json (not imagesTs).
# Saves: predictions, dice CSV, segmentation overlay PNGs -> Drive.
try:
    _run_fold_inference
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

_run_fold_inference(fold=0)


In [ ]:
# -- Cell 15: Fold 1 Training ----------------------------------------------------
try:
    _run_fold_training
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

_run_fold_training(fold=1)

print('\nDrive status after Fold 1 training:')
drive_status()


In [ ]:
# -- Cell 16: Fold 1 Inference + Visualization -----------------------------------
try:
    _run_fold_inference
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

_run_fold_inference(fold=1)


In [ ]:
# -- Cell 17: Fold 2 Training ----------------------------------------------------
try:
    _run_fold_training
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

_run_fold_training(fold=2)

print('\nDrive status after Fold 2 training:')
drive_status()


In [ ]:
# -- Cell 18: Fold 2 Inference + Visualization -----------------------------------
try:
    _run_fold_inference
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

_run_fold_inference(fold=2)


In [ ]:
# -- Cell 19: Fold 3 Training ----------------------------------------------------
try:
    _run_fold_training
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

_run_fold_training(fold=3)

print('\nDrive status after Fold 3 training:')
drive_status()


In [ ]:
# -- Cell 20: Fold 3 Inference + Visualization -----------------------------------
try:
    _run_fold_inference
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

_run_fold_inference(fold=3)


In [ ]:
# -- Cell 21: Fold 4 Training ----------------------------------------------------
try:
    _run_fold_training
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

_run_fold_training(fold=4)

print('\nDrive status after Fold 4 training:')
drive_status()


In [ ]:
# -- Cell 22: Fold 4 Inference + Visualization -----------------------------------
try:
    _run_fold_inference
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

_run_fold_inference(fold=4)


In [ ]:
# -- Cell 22b: Ensemble Inference + External Evaluation on Held-Out Test ---------
# Run AFTER all 5 folds are trained (cells 13,15,17,19,21).

import os, subprocess, sys, shutil
from pathlib import Path

try:
    DRIVE_DIR
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

SEP = '=' * 72
print(SEP)
print('  EXTERNAL HELD-OUT EVALUATION (ENSEMBLE OVER 5 FOLDS)')
print(SEP)

dataset_name = f"Dataset{int(os.environ['DATASET_ID']):03d}_{os.environ['DATASET_NAME']}"
raw_ds_dir   = Path(os.environ['nnUNet_raw']) / dataset_name
images_ts    = raw_ds_dir / 'imagesTs'
labels_ts    = raw_ds_dir / 'labelsTs_gt'
out_pred     = Path('/content/nnunet_work/inference_outputs/held_out_ensemble')
out_pred.mkdir(parents=True, exist_ok=True)

if not images_ts.exists() or not list(images_ts.glob('*_0000.nii.gz')):
    raise RuntimeError('imagesTs is empty. Run Cell 11b first to create held-out external set.')

# Ensure all fold checkpoints exist before ensemble prediction.
num_folds = int(os.environ.get('NUM_FOLDS', '5'))
missing = []
for f in range(num_folds):
    fd = ckpt_dir(f)
    if not ((fd / 'checkpoint_best.pth').exists() or (fd / 'checkpoint_final.pth').exists()):
        missing.append(f)
if missing:
    raise RuntimeError(f'Missing checkpoints for folds: {missing}. Finish training first.')

print(f'Input images (external): {images_ts}')
print(f'Prediction output       : {out_pred}')

# Ensemble inference across all trained folds.
proc = subprocess.Popen(
    [
        sys.executable, '-u', 'scripts/04_inference.py',
        '--input', str(images_ts),
        '--output', str(out_pred),
        '--folds', 'all',
        '--log-dir', '/content/nnunet_work/logs',
    ],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=os.environ.copy(),
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'External ensemble inference failed (rc={proc.returncode})')

preds = sorted(out_pred.glob('*.nii.gz'))
print(f'\nPredictions generated: {len(preds)}')

# Evaluate only if GT labels are available.
has_gt = labels_ts.exists() and bool(list(labels_ts.glob('*.nii.gz')))
if has_gt:
    print('\nRunning external held-out evaluation ...')
    proc = subprocess.Popen(
        [
            sys.executable, '-u', 'scripts/05_evaluate.py',
            '--pred', str(out_pred),
            '--gt', str(labels_ts),
            '--tag', 'held_out_ensemble',
            '--results-dir', '/content/nnunet_work/results',
            '--nsd-tolerance', os.environ['NSD_TOLERANCE_MM'],
            '--low-dice', os.environ['LOW_DICE_THRESHOLD'],
            '--bootstrap-n', os.environ['BOOTSTRAP_N'],
            '--show-rankings',
            '--latex',
            '--log-dir', '/content/nnunet_work/logs',
        ],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy(),
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'External evaluation failed (rc={proc.returncode})')
else:
    print('\nGT labels not found in labelsTs_gt, skipping evaluation (inference-only mode).')

# Save held-out artifacts to Drive.
drive_holdout = DRIVE_DIR / 'held_out_external'
drive_holdout.mkdir(parents=True, exist_ok=True)

pred_dst = drive_holdout / 'predictions'
pred_dst.mkdir(parents=True, exist_ok=True)
for fp in preds:
    shutil.copy2(fp, pred_dst / fp.name)

res_dir = Path('/content/nnunet_work/results')
for rp in sorted(res_dir.glob('held_out_ensemble*')):
    shutil.copy2(rp, drive_holdout / rp.name)

print(f"\nSaved held-out artifacts to Drive: {drive_holdout}")
print(SEP)

In [ ]:
# -- Cell 23: Cross-Fold Summary + Full Evaluation + Drive Save -----------------
# Runs full evaluation (DSC, HD95, NSD, Precision, Recall, Specificity, Volume)
# on all held-out predictions, then prints a MICCAI/CVPR-style table and
# saves everything to Drive.
#
# Prerequisite: run inference cells 14, 16, 18, 20, 22 first.

import subprocess, sys, os, shutil
import pandas as pd, numpy as np
from pathlib import Path

try:
    DRIVE_DIR
except NameError:
    raise RuntimeError('Run Cell 12 (Helper Functions) first.')

SEP  = '=' * 72
SEP2 = '-' * 72

# ══════════════════════════════════════════════════════════════════════════════
# PART 1 — TRAINING SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print(SEP)
print('  PART 1 — TRAINING SUMMARY  (Pseudo Dice = training proxy metric)')
print(SEP)
print(f'  {"Fold":>4}  {"Epochs":>6}  {"Best Pseudo Dice":>16}  {"Final Train Loss":>16}')
print(f'  {"-"*48}')
for fold in range(int(os.environ.get('NUM_FOLDS', 5))):
    rows = parse_training_log(fold)
    if not rows:
        print(f'  {fold:>4}  not trained')
        continue
    valid_dices = [r['pseudo_dice'] for r in rows if r.get('pseudo_dice') is not None]
    best_d  = max(valid_dices) if valid_dices else float('nan')
    final_l = rows[-1].get('train_loss') or float('nan')
    print(f'  {fold:>4}  {len(rows):>6}  {best_d:>16.4f}  {final_l:>16.4f}')

# ══════════════════════════════════════════════════════════════════════════════
# PART 2 — QUICK DICE (in-notebook, computed from predictions)
# ══════════════════════════════════════════════════════════════════════════════
print(f'\n{SEP}')
print('  PART 2 — QUICK DICE  (Dice on held-out val cases, computed in-notebook)')
print(SEP)
print(f'  {"Fold":>4}  {"N Cases":>8}  {"Mean":>8}  {"Std":>7}  {"Min":>7}  {"Max":>7}')
print(f'  {"-"*52}')
all_quick_dices = []
for fold in range(int(os.environ.get('NUM_FOLDS', 5))):
    csv_path = DRIVE_DIR / 'metrics' / f'fold_{fold}_inference_dice.csv'
    if not csv_path.exists():
        print(f'  {fold:>4}  (no inference yet — run Cell {14 + fold*2})')
        continue
    df    = pd.read_csv(str(csv_path))
    dvals = df['dice'].values
    all_quick_dices.extend(dvals)
    print(f'  {fold:>4}  {len(dvals):>8}  {dvals.mean():>8.4f}  {dvals.std():>7.4f}'
          f'  {dvals.min():>7.4f}  {dvals.max():>7.4f}')
if all_quick_dices:
    arr = np.array(all_quick_dices)
    print(f'  {"-"*52}')
    print(f'  {"ALL":>4}  {len(arr):>8}  {arr.mean():>8.4f}  {arr.std():>7.4f}'
          f'  {arr.min():>7.4f}  {arr.max():>7.4f}')

# ══════════════════════════════════════════════════════════════════════════════
# PART 3 — FULL EVALUATION (publication-quality metrics)
# ══════════════════════════════════════════════════════════════════════════════
print(f'\n{SEP}')
print('  PART 3 — FULL EVALUATION  (DSC, HD95, NSD, Precision, Recall, Volume)')
print(SEP)

RESULTS_LOCAL = Path('/content/nnunet_work/results')
RESULTS_LOCAL.mkdir(parents=True, exist_ok=True)

# predictions are at /content/nnunet_work/predictions/fold_N/
# 05_evaluate.py --cv-mode looks for <pred_root>/fold_N/ using --pred flag
PRED_ROOT = Path('/content/nnunet_work/predictions')

trained_folds = [f for f in range(int(os.environ.get('NUM_FOLDS', 5)))
                 if (PRED_ROOT / f'fold_{f}').exists()]

if not trained_folds:
    print('  No predictions found.')
    print('  Run inference cells first: 14 (fold 0), 16 (fold 1), 18 (fold 2),')
    print('  20 (fold 3), 22 (fold 4).')
else:
    print(f'  Folds with predictions: {trained_folds}')
    print('  Running 05_evaluate.py ...\n')

    proc = subprocess.Popen(
        [sys.executable, '-u', 'scripts/05_evaluate.py',
         '--cv-mode',
         '--pred',           str(PRED_ROOT),
         '--results-dir',    str(RESULTS_LOCAL),
         '--nsd-tolerance',  os.environ['NSD_TOLERANCE_MM'],
         '--low-dice',       os.environ['LOW_DICE_THRESHOLD'],
         '--bootstrap-n',    os.environ['BOOTSTRAP_N'],
         '--show-rankings',
         '--latex',
         '--log-dir',        '/content/nnunet_work/logs'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=os.environ.copy()
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    cv_csv = RESULTS_LOCAL / 'cv_combined.csv'
    if cv_csv.exists():
        df_eval = pd.read_csv(str(cv_csv))

        METRIC_DISPLAY = {
            'dice'               : ('DSC',                    True),
            'hd95'               : ('HD95 (mm)',              False),
            'hd'                 : ('HD (mm)',                False),
            'nsd'                : ('NSD',                    True),
            'precision'          : ('Precision',              True),
            'recall'             : ('Recall (Sensitivity)',   True),
            'specificity'        : ('Specificity',            True),
            'volume_similarity'  : ('Volume Similarity',      True),
            'abs_volume_error_ml': ('Abs. Volume Error (mL)', False),
        }

        print(f'\n{SEP}')
        print('  QUANTITATIVE RESULTS — BraTS MEN-RT (5-Fold CV)')
        print(f'  Dataset: {int(os.environ["MAX_TRAIN_CASES"])} training cases  |  '
              f'Folds evaluated: {len(trained_folds)}  |  '
              f'N cases total: {len(df_eval)}')
        print(SEP)
        print(f'  {"Metric":<30}  {"Mean ± Std":>20}  {"Median [IQR]":>24}  Dir')
        print(f'  {"-"*80}')

        summary_rows = []
        for col, (label, higher_better) in METRIC_DISPLAY.items():
            if col not in df_eval.columns:
                continue
            v = pd.to_numeric(df_eval[col], errors='coerce')
            v = v.replace([float('inf'), float('-inf')], float('nan')).dropna()
            if v.empty:
                continue
            mean, std = v.mean(), v.std()
            med       = v.median()
            q25, q75  = v.quantile(0.25), v.quantile(0.75)
            direction = '↑' if higher_better else '↓'
            mean_std  = f'{mean:.4f} ± {std:.4f}'
            med_iqr   = f'{med:.4f} [{q25:.4f},{q75:.4f}]'
            print(f'  {label:<30}  {mean_std:>20}  {med_iqr:>24}  {direction}')
            summary_rows.append({
                'Metric': label, 'Direction': direction,
                'Mean': round(mean, 4), 'Std': round(std, 4),
                'Median': round(med, 4), 'IQR_25': round(q25, 4),
                'IQR_75': round(q75, 4), 'Min': round(v.min(), 4),
                'Max': round(v.max(), 4), 'N': len(v),
            })

        print(f'  {"-"*80}')
        print(f'  N = {len(df_eval)} cases  |  ↑ higher is better  |  ↓ lower is better')
        print(SEP)

        # outliers
        if 'dice' in df_eval.columns:
            thresh = float(os.environ['LOW_DICE_THRESHOLD'])
            outliers = df_eval[pd.to_numeric(df_eval['dice'], errors='coerce') < thresh]
            if not outliers.empty:
                print(f'  Outliers (DSC < {thresh}): {len(outliers)} cases')
                id_col = 'case_id' if 'case_id' in outliers.columns else outliers.columns[0]
                for _, row in outliers.iterrows():
                    print(f'    {row[id_col]}  DSC={row["dice"]:.4f}')

        # ── Save summary CSV to Drive ──
        if summary_rows:
            summary_df = pd.DataFrame(summary_rows)
            dm = DRIVE_DIR / 'metrics'
            dm.mkdir(parents=True, exist_ok=True)
            summary_csv = dm / 'cv_evaluation_summary.csv'
            summary_df.to_csv(str(summary_csv), index=False)
            print(f'\n  Evaluation summary CSV -> Drive: {summary_csv}')

        # ── Copy all result files to Drive ──
        dr = DRIVE_DIR / 'results'
        dr.mkdir(parents=True, exist_ok=True)
        n_copied = 0
        for f in sorted(RESULTS_LOCAL.rglob('*')):
            if f.is_file():
                dst_f = dr / f.name
                shutil.copy2(f, dst_f)
                n_copied += 1
        print(f'  {n_copied} result file(s) copied to Drive/results/')

    else:
        print('  cv_combined.csv not found — evaluation may have failed.')
        print('  Check the output above for errors.')

# ══════════════════════════════════════════════════════════════════════════════
# FINAL DRIVE STATUS
# ══════════════════════════════════════════════════════════════════════════════
print(f'\n{SEP}')
print('  FINAL DRIVE CONTENTS')
print(SEP)
drive_status()
print('\nNext step: use /content/nnunet_work/predictions/ as prompts for your foundation model.')


---
## Resume Instructions

If training was interrupted (GPU limit, session timeout, etc.):

1. Start a new Colab session with **T4 GPU**
2. Run **Cell 1** (install packages)
3. Run **Cell 2** (GPU check)
4. Run **Cell 3** (config)
5. Run **Cell 4** (mount Drive)
6. Run **Cell 5** (Kaggle credentials)
7. Run **Cell 6** (download data — fast, 1-2 min)
8. Run **Cell 7** (clone repo + set env)
9. Run **Cell 8** (register trainer)
10. **Skip Cells 9-11** if data already preprocessed
11. Run **Cell 12** (helper functions)
12. Run the fold cell you want — completed folds auto-skip, incomplete folds resume from `checkpoint_latest.pth`

**To add more epochs to a fold:**
1. Change `NNUNET_NUM_EPOCHS` in Cell 3 to a higher number
2. Re-run Cell 7 to update the env variable
3. Re-run Cell 12 (helpers)
4. Re-run the fold training cell — it will resume from the last checkpoint

**All outputs saved to:**
`Google Drive → My Drive → nnunet_checkpoints/`
- `nnunet_results/` — model checkpoints
- `metrics/` — training + inference CSV files
- `visualizations/` — segmentation overlay PNGs